# 1. Environment Check & Storage Setup (Kaggle Dual T4 / Colab)
Detects whether you are running on **Kaggle** (2x T4 15 GB VRAM = 30 GB total, 30 GB system RAM) or **Colab**, verifies both GPUs are visible (`cuda:0` and `cuda:1`), and on Kaggle prepares `/tmp/comfy_models` so large models (~38 GB total) do not overflow `/kaggle/working`'s ~20 GB cap.

In [ ]:
#@markdown ### Environment & Optional Google Drive Setup
GDRIVE_MODE = "SKIP" #@param ["SKIP", "MOUNT", "UNMOUNT"]

import os
import shutil
import torch

IS_KAGGLE = os.path.exists("/kaggle/working")
BASE_DIR = "/kaggle/working" if IS_KAGGLE else "/content"
MODEL_STORE_DIR = "/tmp/comfy_models" if IS_KAGGLE else None

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Colab / Linux'}")
print(f"Base workspace directory: {BASE_DIR}")
if IS_KAGGLE:
    os.makedirs(MODEL_STORE_DIR, exist_ok=True)
    print(f"Kaggle large-model storage (symlink target): {MODEL_STORE_DIR}")

gpu_count = torch.cuda.device_count()
print(f"Visible CUDA GPUs: {gpu_count}")
for idx in range(gpu_count):
    props = torch.cuda.get_device_properties(idx)
    print(f"  cuda:{idx} -> {props.name} ({props.total_memory / (1024**3):.2f} GB VRAM)")

if gpu_count < 2:
    print("\n⚠️ Fewer than 2 GPUs detected. On Kaggle, select Accelerator: GPU T4 x2 in Session Options.")

if not IS_KAGGLE and GDRIVE_MODE != "SKIP":
    try:
        from google.colab import drive
        if GDRIVE_MODE == "MOUNT":
            drive.mount._DEBUG = False
            drive.mount('/content/drive', force_remount=True)
        elif GDRIVE_MODE == "UNMOUNT":
            try:
                drive.flush_and_unmount()
            except ValueError:
                pass
            get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")
    except ImportError:
        print("google.colab not available; skipping Google Drive.")


# 2. Setup ComfyUI, Dependencies & Custom Nodes (Dual T4 + DisTorch2)
Clones latest ComfyUI, installs core packages (`requirements.txt`, `torchao`, `accelerate`, `xformers`, `gradio`), and installs **`ComfyUI-MultiGPU`** (for **DisTorch2** dual-GPU `.safetensors` sharding across `cuda:0` + `cuda:1`), **`ComfyUI-Manager`**, **`ComfyUI-H3-Modality-Lora-Loader`**, and **`ComfyUI-MiniMax-H3-Turbo`**.

In [ ]:
#@markdown ### Setup ComfyUI & Dual-GPU Nodes
import os
import shutil

if 'BASE_DIR' not in globals():
    IS_KAGGLE = os.path.exists("/kaggle/working")
    BASE_DIR = "/kaggle/working" if IS_KAGGLE else "/content"
    MODEL_STORE_DIR = "/tmp/comfy_models" if IS_KAGGLE else None

DRIVE_PATH = ""  # @param {type:"string"}
UPDATE_COMFY_UI = True  # @param {type:"boolean"}
PULL_GITHUB_CHANGES = True  # @param {type:"boolean"}
INSTALL_COMFY_MANAGER = True  # @param {type:"boolean"}
INSTALL_MULTIGPU = True  # @param {type:"boolean"}
INSTALL_H3_LORA_LOADER = True  # @param {type:"boolean"}

repo_dir = os.path.join(BASE_DIR, "ComfyUI-On-Colab")
if PULL_GITHUB_CHANGES:
    repo_candidates = [repo_dir, "/content/ComfyUI-On-Colab", "/content/drive/MyDrive/ComfyUI-On-Colab"]
    repo_found = False
    for r_dir in repo_candidates:
        if os.path.exists(os.path.join(r_dir, ".git")):
            print(f"-= Pulling latest changes in {r_dir} =-")
            !git -C {r_dir} pull
            repo_found = True
            break
    if not repo_found:
        print(f"-= Cloning ComfyUI-On-Colab repo to {repo_dir} =-")
        !git clone https://github.com/parsamrrelax/ComfyUI-On-Colab.git {repo_dir}

WORKSPACE = os.path.join(BASE_DIR, "ComfyUI")
if DRIVE_PATH.strip():
    WORKSPACE = os.path.join(DRIVE_PATH.strip(), "ComfyUI")

%cd {BASE_DIR}
if not os.path.exists(WORKSPACE):
    print("-= Initial setup: Cloning ComfyUI =-")
    !git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE}

%cd {WORKSPACE}

if UPDATE_COMFY_UI:
    print("-= Updating ComfyUI to latest version =-")
    !git pull

print("-= Installing Core Dependencies =-")
!pip install -q -r requirements.txt
!pip install -q torchao accelerate xformers gradio

# Create / symlink model directories (uses /tmp/comfy_models on Kaggle to avoid 20GB /kaggle/working limit)
model_subdirs = ["diffusion_models", "unet", "text_encoders", "vae", "checkpoints", "loras"]
for sub in model_subdirs:
    local_path = os.path.join(WORKSPACE, "models", sub)
    if IS_KAGGLE and MODEL_STORE_DIR:
        target_path = os.path.join(MODEL_STORE_DIR, sub)
        os.makedirs(target_path, exist_ok=True)
        if os.path.islink(local_path):
            if os.readlink(local_path) != target_path:
                os.unlink(local_path)
                os.symlink(target_path, local_path)
        elif os.path.exists(local_path):
            # Move any existing files before symlinking
            for item in os.listdir(local_path):
                src_item = os.path.join(local_path, item)
                dst_item = os.path.join(target_path, item)
                if not os.path.exists(dst_item):
                    shutil.move(src_item, dst_item)
            shutil.rmtree(local_path, ignore_errors=True)
            os.symlink(target_path, local_path)
        else:
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            os.symlink(target_path, local_path)
    else:
        os.makedirs(local_path, exist_ok=True)

os.makedirs(f"{WORKSPACE}/user/default/workflows", exist_ok=True)
os.makedirs(f"{BASE_DIR}/workflows", exist_ok=True)

def clone_or_update(repo_url, dest_dir, label):
    if os.path.exists(dest_dir):
        existing_remote = !git -C {dest_dir} remote get-url origin
        if not existing_remote or existing_remote[0].strip().rstrip(".git") != repo_url.rstrip(".git"):
            print(f"-= Replacing {label} with {repo_url} =-")
            shutil.rmtree(dest_dir, ignore_errors=True)
    if not os.path.exists(dest_dir):
        print(f"-= Installing {label} =-")
        !git clone {repo_url} {dest_dir}
    else:
        print(f"-= Updating {label} =-")
        !git -C {dest_dir} pull
    req_file = os.path.join(dest_dir, "requirements.txt")
    if os.path.exists(req_file):
        print(f"-= Installing requirements for {label} =-")
        !pip install -q -r {req_file}

if INSTALL_COMFY_MANAGER:
    clone_or_update("https://github.com/ltdrdata/ComfyUI-Manager.git", f"{WORKSPACE}/custom_nodes/ComfyUI-Manager", "ComfyUI-Manager")

if INSTALL_MULTIGPU:
    clone_or_update("https://github.com/pollockjj/ComfyUI-MultiGPU.git", f"{WORKSPACE}/custom_nodes/ComfyUI-MultiGPU", "ComfyUI-MultiGPU (DisTorch2)")

if INSTALL_H3_LORA_LOADER:
    clone_or_update("https://github.com/Dantemss/ComfyUI-H3-Modality-Lora-Loader.git", f"{WORKSPACE}/custom_nodes/ComfyUI-H3-Modality-Lora-Loader", "ComfyUI-H3-Modality-Lora-Loader")
    clone_or_update("https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git", f"{WORKSPACE}/custom_nodes/ComfyUI-MiniMax-H3-Turbo", "ComfyUI-MiniMax-H3-Turbo")

print("\n✅ ComfyUI Dual-T4 setup and custom nodes (including ComfyUI-MultiGPU) are ready!")


# 3. Download MiniMax H3 Dual-T4 Workflows (`16_gb_with_lora_dual.json`)
Installs **`16_gb_with_lora_dual.json`** (configured with **DisTorch2** MultiGPU loaders to shard the ~21 GB diffusion model and ~15.7 GB Qwen3-VL text encoder across `cuda:0` + `cuda:1` with `eject_models=true` so they fit cleanly in your 30 GB combined Dual-T4 VRAM) along with the standard workflows.

In [ ]:
#@markdown ### Download MiniMax H3 Dual-GPU Workflows (with Multi-LoRA Support)
import os
import shutil

if 'BASE_DIR' not in globals():
    BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
if 'WORKSPACE' not in globals():
    WORKSPACE = os.path.join(BASE_DIR, "ComfyUI")

workflows = {
    "16_gb_with_lora_dual.json": "https://raw.githubusercontent.com/parsamrrelax/ComfyUI-On-Colab/main/16_gb_with_lora_dual.json",
    "16gbs_with_lora.json": "https://raw.githubusercontent.com/parsamrrelax/ComfyUI-On-Colab/main/16gbs_with_lora.json",
    "minimax_h3_image_16gb_vram.json": "https://huggingface.co/vyrekai/minimax-h3-image-comfyui-workflows/resolve/main/minimax_h3_image_16gb_vram.json?download=true",
    "minimax_h3_image_12gb_vram.json": "https://huggingface.co/vyrekai/minimax-h3-image-comfyui-workflows/resolve/main/minimax_h3_image_12gb_vram.json?download=true"
}

target_dirs = [
    f"{WORKSPACE}/user/default/workflows",
    f"{BASE_DIR}/workflows"
]

local_repo_candidates = [
    os.path.join(BASE_DIR, "ComfyUI-On-Colab"),
    "/content/ComfyUI-On-Colab",
    "/content/drive/MyDrive/ComfyUI-On-Colab"
]

for name, url in workflows.items():
    local_src = None
    for cand in local_repo_candidates:
        cand_wf = os.path.join(cand, name)
        if os.path.exists(cand_wf):
            local_src = cand_wf
            break
    for t_dir in target_dirs:
        os.makedirs(t_dir, exist_ok=True)
        dest = os.path.join(t_dir, name)
        if local_src:
            shutil.copy2(local_src, dest)
        else:
            !wget -q -nc -O "{dest}" "{url}"
    print(f"✅ Installed workflow: {name}")

print("\n🎉 MiniMax H3 Dual-T4 workflow (16_gb_with_lora_dual.json) is ready!")


# 4. Download MiniMax H3 Models & LoRAs (Aria2 High-Speed Accelerator)
Downloads the MiniMax H3 INT8 ConvRot diffusion model (~21 GB), Qwen3-VL 32B text encoder (~15.7 GB), VAE (~1.2 GB), and optional LoRAs using 16 parallel connections. On Kaggle, these download into `/tmp/comfy_models/` via symlinks so you do not run out of `/kaggle/working` space.

In [ ]:
#@markdown ### MiniMax H3 Models Download Configuration
DOWNLOAD_DIFFUSION_MODEL = True  # @param {type:"boolean"}
#@markdown > **Diffusion Model**: minimax_h3_image_pruned_int8_convrot.safetensors (~21 GB) -> `models/diffusion_models/`

DOWNLOAD_TEXT_ENCODER = True  # @param {type:"boolean"}
#@markdown > **Text Encoder**: qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors (~15.7 GB) -> `models/text_encoders/`

DOWNLOAD_VAE = True  # @param {type:"boolean"}
#@markdown > **VAE**: minimax_h3_image_vae_fp16.safetensors (~1.2 GB) -> `models/vae/`

DOWNLOAD_PROMPT_UPSAMPLER = False  # @param {type:"boolean"}
#@markdown > **Prompt Upsampler (Optional)**: qwen3.5_9b_w4a8_gptq.safetensors (~7.3 GB) -> `models/text_encoders/`

DOWNLOAD_SAMPLE_LORA = False  # @param {type:"boolean"}
#@markdown > **Sample LoRA (Optional)**: fal MiniMax H3 Realism People LoRA -> `models/loras/`

DOWNLOAD_H3_LORAS = True  # @param {type:"boolean"}
#@markdown > **H3 LoRAs**: `largebreastsh3.safetensors`, `femaleanatomyh3.safetensors` -> `models/loras/`

CUSTOM_LORA_URL = ""  # @param {type:"string"}
#@markdown > **Custom LoRA URL (Optional)**: Paste any Hugging Face / Civitai direct URL for MiniMax H3 video/image LoRA

import os
if 'BASE_DIR' not in globals():
    BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
if 'WORKSPACE' not in globals():
    WORKSPACE = os.path.join(BASE_DIR, "ComfyUI")

!apt-get -y install -qq aria2

def aria_download(url, dest_dir, filename):
    os.makedirs(dest_dir, exist_ok=True)
    real_dir = os.path.realpath(dest_dir)
    target_path = os.path.join(real_dir, filename)
    if os.path.exists(target_path) and os.path.getsize(target_path) > 1024 * 1024:
        print(f"⏩ {filename} already exists ({os.path.getsize(target_path)/(1024**3):.2f} GB), skipping download.")
        return
    print(f"🚀 Downloading {filename} to {real_dir}...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{real_dir}" -o "{filename}"
    print(f"✅ Finished: {filename}")

if DOWNLOAD_DIFFUSION_MODEL:
    aria_download(
        "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/diffusion_models/minimax_h3_image_pruned_int8_convrot.safetensors?download=true",
        f"{WORKSPACE}/models/diffusion_models",
        "minimax_h3_image_pruned_int8_convrot.safetensors"
    )

if DOWNLOAD_TEXT_ENCODER:
    aria_download(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors?download=true",
        f"{WORKSPACE}/models/text_encoders",
        "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
    )

if DOWNLOAD_VAE:
    aria_download(
        "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/vae/minimax_h3_image_vae_fp16.safetensors?download=true",
        f"{WORKSPACE}/models/vae",
        "minimax_h3_image_vae_fp16.safetensors"
    )

if DOWNLOAD_PROMPT_UPSAMPLER:
    aria_download(
        "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/text_encoders/qwen3.5_9b_w4a8_gptq.safetensors?download=true",
        f"{WORKSPACE}/models/text_encoders",
        "qwen3.5_9b_w4a8_gptq.safetensors"
    )

if DOWNLOAD_SAMPLE_LORA:
    aria_download(
        "https://huggingface.co/fal/MiniMax-H3-Realism-People-LoRA/resolve/main/minimax_h3_realism_lora.safetensors",
        f"{WORKSPACE}/models/loras",
        "minimax_h3_realism_lora.safetensors"
    )

if DOWNLOAD_H3_LORAS:
    h3_loras = [
        ("largebreastsh3.safetensors", "https://huggingface.co/Mrrelax/pp/resolve/main/largebreastsh3.safetensors?download=true"),
        ("femaleanatomyh3.safetensors", "https://huggingface.co/Mrrelax/pp/resolve/main/femaleanatomyh3.safetensors?download=true"),
    ]
    for lora_name, lora_url in h3_loras:
        aria_download(lora_url, f"{WORKSPACE}/models/loras", lora_name)

if CUSTOM_LORA_URL.strip():
    custom_name = CUSTOM_LORA_URL.strip().split('/')[-1].split('?')[0]
    aria_download(
        CUSTOM_LORA_URL.strip(),
        f"{WORKSPACE}/models/loras",
        custom_name
    )

print("\n🎉 MiniMax H3 models and LoRAs are completely downloaded and verified!")


# (Optional) Download Extra Checkpoints / CivitAI Models
Optional helper if you wish to download additional models or CivitAI checkpoints into `models/checkpoints/`.

In [ ]:
#@markdown ### Optional: Download Extra Checkpoints
DOWNLOAD_EXTRA_MODELS = False  # @param {type:"boolean"}

if DOWNLOAD_EXTRA_MODELS:
    import os
    if 'BASE_DIR' not in globals():
        BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
    if 'WORKSPACE' not in globals():
        WORKSPACE = os.path.join(BASE_DIR, "ComfyUI")

    CIVITAI_API_TOKEN = None
    try:
        from kaggle_secrets import UserSecretsClient
        CIVITAI_API_TOKEN = UserSecretsClient().get_secret("CIVITAI_API_TOKEN")
    except Exception:
        try:
            from google.colab import userdata
            CIVITAI_API_TOKEN = userdata.get("CIVITAI_API_TOKEN")
        except Exception:
            CIVITAI_API_TOKEN = os.environ.get("CIVITAI_API_TOKEN")

    print("Loaded CivitAI API key:", "✅" if CIVITAI_API_TOKEN else "❌ Not found (public downloads only)")

    def downloadCheckpoint(url, filename=None):
        dest = os.path.realpath(f"{WORKSPACE}/models/checkpoints")
        os.makedirs(dest, exist_ok=True)
        if "huggingface.co" in url:
            if not filename:
                filename = url.split("/")[-1].removesuffix("?download=true")
            !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{dest}" -o "{filename}"
        else:
            token_suffix = f"?token={CIVITAI_API_TOKEN}" if CIVITAI_API_TOKEN else ""
            full_url = f"{url}{token_suffix}"
            if filename:
                !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{full_url}" -d "{dest}" -o "{filename}"
            else:
                !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{full_url}" -d "{dest}"

    # Example: downloadCheckpoint("https://civitai.com/api/download/models/2071650", "cyberrealisticPony.safetensors")
else:
    print("Skipping extra checkpoints download.")


# 5. Verify File Structure, Disk Space & Dual GPUs
Inspect installed models, custom nodes (`ComfyUI-MultiGPU`), workflows (`16_gb_with_lora_dual.json`), and disk space before starting the server.

In [ ]:
import os
if 'BASE_DIR' not in globals():
    BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
if 'WORKSPACE' not in globals():
    WORKSPACE = os.path.join(BASE_DIR, "ComfyUI")

print("=== Disk Usage ===")
!df -h /tmp {BASE_DIR}

print("\n=== Diffusion Models (models/diffusion_models/) ===")
!ls -lhL {WORKSPACE}/models/diffusion_models/

print("\n=== Text Encoders (models/text_encoders/) ===")
!ls -lhL {WORKSPACE}/models/text_encoders/

print("\n=== VAE Models (models/vae/) ===")
!ls -lhL {WORKSPACE}/models/vae/

print("\n=== LoRA Models (models/loras/) ===")
!ls -lhL {WORKSPACE}/models/loras/

print("\n=== Custom Nodes (custom_nodes/) ===")
!ls -1 {WORKSPACE}/custom_nodes/

print("\n=== Workflows (user/default/workflows/) ===")
!ls -lh {WORKSPACE}/user/default/workflows/


# 6. Start ComfyUI & Expose Server (Dual-T4 Tuned Flags)
Launches ComfyUI **without** `--cuda-device` (exposing both `cuda:0` and `cuda:1` to `ComfyUI-MultiGPU`) and **without** `--highvram` (preventing 15 GB per-card OOM crashes):
- `--enable-dynamic-vram`: Enables dynamic weight streaming/offloading for any residual allocations.
- `--reserve-vram 1.5`: Keeps 1.5 GB VRAM headroom per GPU for latent sampling and decoding.
- `--cache-ram 6`: Evicts cached items before Kaggle's 30 GB system RAM limit is hit.
- Optional `LOW_RAM_SAFE_MODE` (`--cache-none --disable-pinned-memory`): Enable only if you still hit 30 GB system RAM limits.

In [ ]:
#@markdown ### Start ComfyUI (Dual T4 + 30GB RAM Tuned)
TUNNEL_TYPE = "Gradio" #@param ["Gradio", "Cloudflare", "Pinggy", "Ngrok"]
#@markdown > **Gradio (Recommended)**: Fast, unthrottled downloads for large (16MP) images without token setup.
#@markdown > **Cloudflare**: Standard `trycloudflare.com` quick tunnel.
#@markdown > **Pinggy**: Free SSH-based high-speed tunnel (`pinggy.link`).
#@markdown > **Ngrok**: Dedicated tunnel (paste token below if selected).
NGROK_TOKEN = "" #@param {type:"string"}
RESERVE_VRAM_GB = 1.5 #@param {type:"number"}
CACHE_RAM_GB = 6 #@param {type:"number"}
LOW_RAM_SAFE_MODE = False #@param {type:"boolean"}
#@markdown > Enable `LOW_RAM_SAFE_MODE` (`--cache-none --disable-pinned-memory`) only if you hit Kaggle's 30 GB system RAM limit.

import subprocess
import threading
import time
import socket
import re
import os
import secrets

if 'BASE_DIR' not in globals():
    BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
if 'WORKSPACE' not in globals():
    WORKSPACE = os.path.join(BASE_DIR, "ComfyUI")

%cd {WORKSPACE}

if TUNNEL_TYPE == "Gradio":
    try:
        import gradio
    except ImportError:
        print("Installing gradio for high-speed tunneling...")
        !pip install -q gradio
elif TUNNEL_TYPE == "Ngrok":
    try:
        import pyngrok
    except ImportError:
        print("Installing pyngrok...")
        !pip install -q pyngrok

def wait_for_port(port=8188):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(("127.0.0.1", port))
        sock.close()
        if result == 0:
            break

def print_banner(tunnel_name, url):
    print("\n" + "=" * 75)
    print(f"🚀 ComfyUI {tunnel_name} URL: {url}")
    print("=" * 75)
    print("💡 Tips:")
    print(" - Load workflow: 16_gb_with_lora_dual.json (DisTorch2 Dual-T4 + Multi-LoRA)")
    print(" - Both T4 GPUs (cuda:0 + cuda:1) are exposed for DisTorch2 layer sharding.")
    print(" - LoRA nodes are bypassed by default (Ctrl+B on the LoRA node to activate).\n")

def launch_cloudflare(port=8188):
    if not os.path.exists("/usr/local/bin/cloudflared"):
        print("Installing Cloudflared binary...")
        !curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
        !chmod +x /usr/local/bin/cloudflared

    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )
    tunnel_url = None
    for line in p.stderr:
        if not tunnel_url:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if match:
                tunnel_url = match.group(0)
                print_banner("Cloudflare", tunnel_url)

def create_gradio_tunnel(setup_tunnel, host, port, share_token):
    attempts = [
        lambda: setup_tunnel(host, port, share_token, None, None),
        lambda: setup_tunnel(host, port, share_token, None),
        lambda: setup_tunnel(host, port, share_token),
    ]
    last_err = None
    for attempt in attempts:
        try:
            return attempt()
        except (TypeError, ValueError) as e:
            last_err = e
            continue
    raise last_err

def launch_tunnel(tunnel_type, port=8188):
    wait_for_port(port)
    print(f"\n[Tunnel] ComfyUI server is up! Establishing {tunnel_type} tunnel...")

    if tunnel_type == "Gradio":
        try:
            try:
                from gradio.networking import setup_tunnel
            except ImportError:
                try:
                    from gradio.tunneling import setup_tunnel
                except ImportError:
                    from gradio.blocks import setup_tunnel
            share_token = secrets.token_urlsafe(32)
            public_url = create_gradio_tunnel(setup_tunnel, "127.0.0.1", port, share_token)
            print_banner("Gradio (High-Speed)", public_url)
            while True:
                time.sleep(1)
        except Exception as e:
            print(f"\n⚠️ Gradio tunnel could not start ({e}). Falling back to Cloudflare Tunnel...")
            launch_cloudflare(port)

    elif tunnel_type == "Cloudflare":
        launch_cloudflare(port)
    elif tunnel_type == "Pinggy":
        p = subprocess.Popen(
            ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30", "-p", "443", f"-R0:localhost:{port}", "a.pinggy.io"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1
        )
        tunnel_url = None
        for line in p.stdout:
            if not tunnel_url:
                match = re.search(r"https://[a-zA-Z0-9-]+\.a\.pinggy\.link", line)
                if match:
                    tunnel_url = match.group(0)
                    print_banner("Pinggy (High-Speed)", tunnel_url)

    elif tunnel_type == "Ngrok":
        from pyngrok import ngrok
        if NGROK_TOKEN.strip():
            ngrok.set_auth_token(NGROK_TOKEN.strip())
        public_url = ngrok.connect(port).public_url
        print_banner("Ngrok (High-Speed)", public_url)
        while True:
            time.sleep(1)

threading.Thread(target=launch_tunnel, daemon=True, args=(TUNNEL_TYPE, 8188)).start()

# Build launch flags (exposes both T4s; avoids --highvram OOM on 15GB/card)
flags = [
    "--listen 127.0.0.1",
    "--port 8188",
    "--dont-print-server",
    "--enable-dynamic-vram",
    f"--reserve-vram {RESERVE_VRAM_GB}",
]
if LOW_RAM_SAFE_MODE:
    flags.extend(["--cache-none", "--disable-pinned-memory"])
else:
    flags.append(f"--cache-ram {CACHE_RAM_GB}")

launch_cmd = f"PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python main.py {' '.join(flags)}"
print(f"Launching ComfyUI: {launch_cmd}")
!{launch_cmd}
